# Shared training definitions
Run through your model notebook. This defines the common loader, model factories and scoring; it performs no training. All labels are derived proxies.

In [ ]:
from pathlib import Path
import os
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
assert (ROOT / "data").is_dir(), "Open this notebook from the repository root or notebooks directory"
os.chdir(ROOT)


In [ ]:
"""Shared loader: IDs and derived-label diagnostics never enter X."""
from pathlib import Path
import numpy as np

def load_training(root='processed', validation_fold=None):
    """Return fit and validation dictionaries; validation retains ALL observations.

    X has shape (observations, candidates, features). Flatten candidate rows for
    binary classifier fitting; reshape positive probabilities for segment choice.
    Masks restrict fitting, never silently remove evaluation failures.
    """
    def collect(files, fitting):
        rows=[]
        for p in files:
            with np.load(p,allow_pickle=False) as d:
                mask=d['training_observation_mask'] if fitting else np.ones(len(d['X']),dtype=bool)
                rows.append((d['X'][mask],d['y'][mask],d['candidate_arc_ids'][mask],d['reference_arc_id'][mask],np.full(mask.sum(),p.stem),d['label_reliable'][mask]))
        if not rows:raise ValueError('No trips found')
        return dict(zip(['X','y','candidate_arc_ids','reference_arc_id','trip_id','label_reliable'],[np.concatenate(x) for x in zip(*rows)]))
    train=[];val=[]
    for p in sorted((Path(root)/'train').glob('*.npz')):
        with np.load(p,allow_pickle=False) as d: fold=int(d['fold'])
        (val if fold==validation_fold else train).append(p)
    if validation_fold is None: val=sorted((Path(root)/'student_test').glob('*.npz'))
    return collect(train,True),collect(val,False)


In [ ]:
"""Shared initial model comparisons. No tuning or HMM is performed here."""
from pathlib import Path
import os
os.environ.setdefault("OMP_NUM_THREADS","2")
os.environ.setdefault("OPENBLAS_NUM_THREADS","2")
import argparse,json,time,platform,hashlib,sys
import numpy as np

SEED=212

def make_model(name):
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import HistGradientBoostingClassifier,RandomForestClassifier,ExtraTreesClassifier
    from sklearn.neural_network import MLPClassifier
    models={
        'logistic':lambda:make_pipeline(StandardScaler(),LogisticRegression(C=1.,max_iter=1000,random_state=SEED)),
        'hist_gradient_boosting':lambda:HistGradientBoostingClassifier(max_iter=150,learning_rate=.1,max_leaf_nodes=31,l2_regularization=1.,random_state=SEED),
        'random_forest':lambda:RandomForestClassifier(n_estimators=150,max_depth=24,min_samples_leaf=2,n_jobs=2,random_state=SEED),
        'extra_trees':lambda:ExtraTreesClassifier(n_estimators=150,max_depth=24,min_samples_leaf=2,n_jobs=2,random_state=SEED),
        # No internal random-row validation split: use the supplied trip folds.
        'mlp':lambda:make_pipeline(StandardScaler(),MLPClassifier(hidden_layer_sizes=(64,32),max_iter=100,early_stopping=False,random_state=SEED)),
    }
    return models[name]()

def score_predictions(data,positive_probability):
    """Arc Macro F1 uses (trip,arc) labels; misses are included in all-point scores."""
    p=np.asarray(positive_probability,dtype=float).reshape(data['y'].shape)
    if not(np.isfinite(p).all() and np.all((p>=0)&(p<=1))):raise ValueError('Invalid probabilities')
    selected=np.argmax(p,axis=1);pred=data['candidate_arc_ids'][np.arange(len(p)),selected]
    truth=data['reference_arc_id'];correct=pred==truth;hit=data['y'].any(axis=1)
    # Tiny explicit none-of-candidates mass prevents dropping candidate misses.
    eps=1e-12;weights=np.maximum(p,eps);denom=weights.sum(axis=1)+eps
    true_probability=np.where(hit,(weights*data['y']).sum(axis=1)/denom,eps/denom)
    def metrics(mask):
        if not mask.any():return None
        trip=data['trip_id'][mask];t=truth[mask];q=pred[mask]
        # Compute multiclass macro metrics without inventing globally shared arc IDs.
        key=lambda arc:np.char.add(np.char.add(trip.astype(str),':'),arc.astype(str))
        tk=key(t);pk=key(q);classes=np.unique(np.r_[tk,pk])
        ti=np.searchsorted(classes,tk);pi=np.searchsorted(classes,pk)
        tp=np.bincount(ti[ti==pi],minlength=len(classes));tc=np.bincount(ti,minlength=len(classes));pc=np.bincount(pi,minlength=len(classes))
        precision=np.divide(tp,pc,out=np.zeros(len(classes)),where=pc>0);recall=np.divide(tp,tc,out=np.zeros(len(classes)),where=tc>0)
        f1=np.divide(2*tp,tc+pc,out=np.zeros(len(classes)),where=(tc+pc)>0)
        return dict(observations=int(mask.sum()),top1_accuracy=float(correct[mask].mean()),macro_precision=float(precision.mean()),macro_recall=float(recall.mean()),macro_f1=float(f1.mean()),candidate_miss_rate=float((~hit[mask]).mean()),categorical_log_loss_with_miss_state=float(-np.log(true_probability[mask]).mean()))
    allpoints=np.ones(len(p),dtype=bool)
    binary_p=np.clip(p,eps,1-eps);y=data['y']
    result=dict(all_points=metrics(allpoints),reliable_points=metrics(data['label_reliable']),reliable_coverage=float(data['label_reliable'].mean()),candidate_binary_log_loss=float(-(y*np.log(binary_p)+(1-y)*np.log(1-binary_p)).mean()),route_continuity=None,route_continuity_reason='Needs raw road network and an agreed path/connectivity definition; do not invent a value.')
    return result,pred

def run(model_name,root,out,folds,final=False,smoke=False):
    import sklearn,joblib,warnings
    out=Path(out)/model_name
    if out.exists() and any(out.iterdir()):raise FileExistsError(f'{out} already contains results; choose a new --out')
    out.mkdir(parents=True,exist_ok=True)
    records=[]
    config=make_model(model_name).get_params(deep=True)
    (out/'configuration.json').write_text(json.dumps(config,indent=2,default=str))
    schema=json.loads((Path(root)/'schema.json').read_text())
    provenance=dict(model=model_name,seed=SEED,python=platform.python_version(),numpy=np.__version__,sklearn=sklearn.__version__,schema=schema,smoke=smoke,argv=sys.argv,dataset_hashes={str(p.relative_to(root)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(Path(root).rglob('*.npz'))})
    (out/'provenance.json').write_text(json.dumps(provenance,indent=2))
    splits=[(f'fold_{fold}',fold) for fold in folds]+([('student_test',None)] if final else [])
    for label,fold in splits:
        train,validation=load_training(root,fold)
        if smoke:
            # A runtime check only; never report these as full experiment results.
            for data in [train,validation]:
                for key in data:data[key]=data[key][:200]
        model=make_model(model_name);X=train['X'].reshape(-1,train['X'].shape[-1]);y=train['y'].ravel()
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always');start=time.perf_counter();model.fit(X,y);fit_s=time.perf_counter()-start
        start=time.perf_counter();prob=model.predict_proba(validation['X'].reshape(-1,validation['X'].shape[-1]))[:,list(model.classes_).index(1)];inference_s=time.perf_counter()-start
        metrics,pred=score_predictions(validation,prob)
        training_probability=model.predict_proba(X)[:,list(model.classes_).index(1)];training_metrics,_=score_predictions(train,training_probability)
        record=dict(split=label,fit_seconds=fit_s,inference_seconds=inference_s,inference_ms_per_observation=inference_s/len(pred)*1000,metrics=metrics,fit_subset_metrics=training_metrics,warnings=[str(w.message) for w in caught],smoke=smoke)
        records.append(record);(out/'metrics.json').write_text(json.dumps(records,indent=2))
        np.savez_compressed(out/f'{label}_predictions.npz',positive_probability=prob.reshape(validation['y'].shape),predicted_arc_id=pred,reference_arc_id=validation['reference_arc_id'],candidate_arc_ids=validation['candidate_arc_ids'],trip_id=validation['trip_id'],label_reliable=validation['label_reliable'])
        if final and fold is None:joblib.dump(model,out/'model.joblib')
        print(label,json.dumps(record),flush=True)
    cv=[r['metrics']['all_points']['macro_f1'] for r in records if r['split'].startswith('fold_')]
    (out/'summary.json').write_text(json.dumps(dict(model=model_name,smoke=smoke,cv_macro_f1_mean=float(np.mean(cv)) if cv else None,cv_macro_f1_std=float(np.std(cv)) if cv else None,completed_folds=len(cv),student_test_completed=any(r['split']=='student_test' for r in records)),indent=2))

